# Week06 HW03

- 데이터셋: Kaggle Titanic (`train.csv`)
- 문제 유형: 이진 분류
- 타깃 변수: `Survived`

### 과제 목표
- Random Forest, XGBoost, LightGBM 성능 비교
- 최소 2개 이상의 핵심 하이퍼파라미터 튜닝
- validation 기반으로 모델 평가
- 최소 1개의 boosting 모델에 early stopping 적용
- 성능 비교와 함께 실험 설계 이유 및 결과 해석 정리

In [ ]:
# 필요할 때만 실행
%pip install -q xgboost lightgbm

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from xgboost import XGBClassifier
import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. 데이터 불러오기

이번 과제에서는 Kaggle Titanic의 `train.csv`를 사용한다.  
test 파일을 따로 쓰지 않고, 하나의 train 데이터 안에서 직접:

- train
- validation
- test

로 분리하여 실험.

In [4]:
df = pd.read_csv("../data/titanic.csv")
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. 사용할 변수 선택

이번 과제에서는 최소한의 전처리로 진행하기 위해 아래 변수 사용.

### 입력 변수
- 수치형:
  - `Pclass`
  - `Age`
  - `SibSp`
  - `Parch`
  - `Fare`

- 범주형:
  - `Sex`
  - `Embarked`

### 타깃 변수
- `Survived`

Tree 계열 모델은 스케일링에 크게 민감하지 않기 때문에  
이번 실험에서는 별도의 scaling 없이 진행한다.

In [7]:
target_col = "Survived"

feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"
]

numeric_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Sex", "Embarked"]

data = df[feature_cols + [target_col]].copy()
data.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0


## 3. Train / Validation / Test Split

- Train: 60%
- Validation: 20%
- Test: 20%

또한 분류 문제이므로 클래스 비율이 크게 흔들리지 않도록  
`stratify` 사용.

- train: 모델 학습
- validation: 하이퍼파라미터 비교 및 early stopping 기준
- test: 최종 모델의 일반화 성능 확인

In [8]:
X = data[feature_cols].copy()
y = data[target_col].copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp,
    test_size=0.25,   # 0.25 * 0.80 = 0.20
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, y_train.shape)
print("Valid shape:", X_valid.shape, y_valid.shape)
print("Test shape :", X_test.shape, y_test.shape)

Train shape: (534, 7) (534,)
Valid shape: (178, 7) (178,)
Test shape : (179, 7) (179,)


## 4. 최소 전처리

### 수치형 변수
- 결측치는 train 기준 median으로 대체

### 범주형 변수
- 결측치는 train 기준 mode로 대체
- one-hot encoding 적용
  
validation/test 정보가 학습 과정에 새어 들어가지 않도록 전처리 기준값을 **반드시 train 데이터에서만 계산**해야 한다. 

In [9]:
def preprocess_titanic(X_train, X_valid, X_test, numeric_cols, categorical_cols):
    X_train = X_train.copy()
    X_valid = X_valid.copy()
    X_test = X_test.copy()

    # train 데이터 기준으로 결측 대체값 계산
    train_num_fill = X_train[numeric_cols].median()
    train_cat_fill = X_train[categorical_cols].mode().iloc[0]

    for X_part in [X_train, X_valid, X_test]:
        X_part[numeric_cols] = X_part[numeric_cols].fillna(train_num_fill)
        X_part[categorical_cols] = X_part[categorical_cols].fillna(train_cat_fill)

    # 범주형 변수 one-hot encoding
    X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=False)
    X_valid = pd.get_dummies(X_valid, columns=categorical_cols, drop_first=False)
    X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=False)

    # train 기준으로 컬럼 정렬 맞추기
    X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    return (
        X_train.astype(float),
        X_valid.astype(float),
        X_test.astype(float),
        train_num_fill,
        train_cat_fill,
    )

X_train_prep, X_valid_prep, X_test_prep, num_fill_values, cat_fill_values = preprocess_titanic(
    X_train, X_valid, X_test, numeric_cols, categorical_cols
)

print(X_train_prep.shape, X_valid_prep.shape, X_test_prep.shape)
X_train_prep.head()

(534, 10) (178, 10) (179, 10)


,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
577,1.0,39.0,1.0,0.0,55.9000,1.0,0.0,0.0,0.0,1.0
63,3.0,4.0,3.0,2.0,27.9000,0.0,1.0,0.0,0.0,1.0
424,3.0,18.0,1.0,1.0,20.2125,0.0,1.0,0.0,0.0,1.0
513,1.0,54.0,1.0,0.0,59.4000,1.0,0.0,1.0,0.0,0.0
610,3.0,39.0,1.0,5.0,31.2750,1.0,0.0,0.0,0.0,1.0


## 5. 평가 함수 정의

- Accuracy: 전체적으로 얼마나 맞췄는지 보기 쉬움
- F1-score: precision / recall 균형을 반영
- ROC-AUC: threshold에 덜 민감해서 모델 비교에 유리


In [10]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
    }

def evaluate_train_valid(model, X_train, y_train, X_valid, y_valid, model_name):
    train_prob = model.predict_proba(X_train)[:, 1]
    valid_prob = model.predict_proba(X_valid)[:, 1]

    train_metrics = compute_metrics(y_train, train_prob)
    valid_metrics = compute_metrics(y_valid, valid_prob)

    row = {
        "Model": model_name,
        "Train_Accuracy": train_metrics["Accuracy"],
        "Train_F1": train_metrics["F1"],
        "Train_ROC_AUC": train_metrics["ROC_AUC"],
        "Valid_Accuracy": valid_metrics["Accuracy"],
        "Valid_F1": valid_metrics["F1"],
        "Valid_ROC_AUC": valid_metrics["ROC_AUC"],
    }
    return row

def evaluate_test(model, X_test, y_test, model_name):
    test_prob = model.predict_proba(X_test)[:, 1]
    test_metrics = compute_metrics(y_test, test_prob)

    return {
        "Model": model_name,
        "Test_Accuracy": test_metrics["Accuracy"],
        "Test_F1": test_metrics["F1"],
        "Test_ROC_AUC": test_metrics["ROC_AUC"],
    }

## 6. Random Forest Baseline

Random Forest:
- 여러 트리를 병렬적으로 학습
- 예측을 평균/다수결로 결합
- 주로 variance 감소에 강한 모델

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_prep, y_train)

rf_result = evaluate_train_valid(
    rf_model, X_train_prep, y_train, X_valid_prep, y_valid, "RandomForest_baseline"
)

pd.DataFrame([rf_result])

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC
0,RandomForest_baseline,0.882022,0.830189,0.93726,0.820225,0.75,0.890575


## 7. XGBoost Baseline

XGBoost:
- boosting 기반으로 오차를 순차적으로 줄여 나감.
- 정규화와 실용적인 구현이 강점
- validation 기반 early stopping 적용

In [12]:
xgb_base = XGBClassifier(
    objective="binary:logistic",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
    verbosity=0
)

xgb_base.fit(
    X_train_prep, y_train,
    eval_set=[(X_valid_prep, y_valid)],
    verbose=False
)

xgb_base_result = evaluate_train_valid(
    xgb_base, X_train_prep, y_train, X_valid_prep, y_valid, "XGBoost_baseline"
)

xgb_base_result["Best_iteration"] = xgb_base.best_iteration
pd.DataFrame([xgb_base_result])

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,Best_iteration
0,XGBoost_baseline,0.895131,0.851852,0.952191,0.837079,0.778626,0.898529,125


## 8. LightGBM Baseline
LightGBM:
- histogram 기반 학습
- leaf-wise 성장 전략 사용
- 속도 및 효율성 좋음
- 과적합 제어 중요

validation 기반 early stopping을 적용

In [13]:
lgb_base = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

lgb_base.fit(
    X_train_prep, y_train,
    eval_set=[(X_valid_prep, y_valid)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

lgb_base_result = evaluate_train_valid(
    lgb_base, X_train_prep, y_train, X_valid_prep, y_valid, "LightGBM_baseline"
)

lgb_base_result["Best_iteration"] = lgb_base.best_iteration_
pd.DataFrame([lgb_base_result])

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,Best_iteration
0,LightGBM_baseline,0.878277,0.826667,0.950463,0.825843,0.763359,0.891711,64


## 9. Baseline 성능 비교

Random Forest vs XGBoost vs LightGBM
- train 성능이 지나치게 높은가
- validation 성능은 어떤가
- boosting 모델이 RF보다 실제로 나은가

In [14]:
baseline_results = pd.DataFrame([
    rf_result,
    xgb_base_result,
    lgb_base_result
]).sort_values("Valid_ROC_AUC", ascending=False)

baseline_results

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,Best_iteration
1,XGBoost_baseline,0.895131,0.851852,0.952191,0.837079,0.778626,0.898529,125.0
2,LightGBM_baseline,0.878277,0.826667,0.950463,0.825843,0.763359,0.891711,64.0
0,RandomForest_baseline,0.882022,0.830189,0.937260,0.820225,0.750000,0.890575,NaN


## 10. XGBoost 튜닝

In [15]:
xgb_param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5],
}

xgb_tuning_rows = []
best_xgb_model = None
best_xgb_params = None
best_xgb_valid_auc = -np.inf

for params in ParameterGrid(xgb_param_grid):
    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1000,
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        verbosity=0
    )

    model.fit(
        X_train_prep, y_train,
        eval_set=[(X_valid_prep, y_valid)],
        verbose=False
    )

    row = evaluate_train_valid(
        model, X_train_prep, y_train, X_valid_prep, y_valid,
        model_name=f"XGBoost_tuned_lr{params['learning_rate']}_depth{params['max_depth']}"
    )
    row["learning_rate"] = params["learning_rate"]
    row["max_depth"] = params["max_depth"]
    row["Best_iteration"] = model.best_iteration
    xgb_tuning_rows.append(row)

    if row["Valid_ROC_AUC"] > best_xgb_valid_auc:
        best_xgb_valid_auc = row["Valid_ROC_AUC"]
        best_xgb_model = model
        best_xgb_params = params

xgb_tuning_df = pd.DataFrame(xgb_tuning_rows).sort_values("Valid_ROC_AUC", ascending=False)
xgb_tuning_df.head(10)

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,learning_rate,max_depth,Best_iteration
6,XGBoost_tuned_lr0.1_depth3,0.900749,0.863049,0.952643,0.859551,0.812030,0.908155,0.10,3,117
3,XGBoost_tuned_lr0.05_depth3,0.874532,0.823219,0.932004,0.831461,0.769231,0.902674,0.05,3,134
7,XGBoost_tuned_lr0.1_depth4,0.887640,0.842932,0.944925,0.825843,0.763359,0.900869,0.10,4,54
8,XGBoost_tuned_lr0.1_depth5,0.908240,0.871391,0.961317,0.859551,0.814815,0.900668,0.10,5,55
0,XGBoost_tuned_lr0.03_depth3,0.878277,0.828496,0.938461,0.831461,0.769231,0.898663,0.03,3,251
4,XGBoost_tuned_lr0.05_depth4,0.895131,0.851852,0.952191,0.837079,0.778626,0.898529,0.05,4,125
5,XGBoost_tuned_lr0.05_depth5,0.910112,0.873684,0.966869,0.842697,0.791045,0.897193,0.05,5,119
1,XGBoost_tuned_lr0.03_depth4,0.889513,0.841823,0.944407,0.820225,0.753846,0.896123,0.03,4,163
2,XGBoost_tuned_lr0.03_depth5,0.902622,0.862434,0.956668,0.837079,0.781955,0.894251,0.03,5,147


## 11. LightGBM 튜닝

In [16]:
lgb_param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 5, 7],
}

lgb_tuning_rows = []
best_lgb_model = None
best_lgb_params = None
best_lgb_valid_auc = -np.inf

for params in ParameterGrid(lgb_param_grid):
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=2000,
        learning_rate=params["learning_rate"],
        num_leaves=params["num_leaves"],
        max_depth=params["max_depth"],
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_prep, y_train,
        eval_set=[(X_valid_prep, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    row = evaluate_train_valid(
        model, X_train_prep, y_train, X_valid_prep, y_valid,
        model_name=f"LGBM_tuned_lr{params['learning_rate']}_leaves{params['num_leaves']}_depth{params['max_depth']}"
    )
    row["learning_rate"] = params["learning_rate"]
    row["num_leaves"] = params["num_leaves"]
    row["max_depth"] = params["max_depth"]
    row["Best_iteration"] = model.best_iteration_
    lgb_tuning_rows.append(row)

    if row["Valid_ROC_AUC"] > best_lgb_valid_auc:
        best_lgb_valid_auc = row["Valid_ROC_AUC"]
        best_lgb_model = model
        best_lgb_params = params

lgb_tuning_df = pd.DataFrame(lgb_tuning_rows).sort_values("Valid_ROC_AUC", ascending=False)
lgb_tuning_df.head(10)

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,learning_rate,num_leaves,max_depth,Best_iteration
12,LGBM_tuned_lr0.05_leaves15_depth5,0.885768,0.839050,0.948506,0.853933,0.800000,0.903743,0.05,15,5,117
21,LGBM_tuned_lr0.1_leaves15_depth5,0.891386,0.848168,0.957469,0.848315,0.796992,0.902941,0.10,15,5,73
4,LGBM_tuned_lr0.03_leaves31_depth5,0.882022,0.833773,0.943932,0.842697,0.781250,0.902674,0.03,31,5,173
5,LGBM_tuned_lr0.03_leaves63_depth5,0.882022,0.833773,0.943932,0.842697,0.781250,0.902674,0.03,63,5,173
26,LGBM_tuned_lr0.1_leaves63_depth7,0.887640,0.841270,0.948469,0.859551,0.809160,0.900468,0.10,63,7,41
25,LGBM_tuned_lr0.1_leaves31_depth7,0.887640,0.841270,0.948469,0.859551,0.809160,0.900468,0.10,31,7,41
14,LGBM_tuned_lr0.05_leaves63_depth5,0.883895,0.835979,0.947439,0.842697,0.781250,0.900267,0.05,63,5,110
13,LGBM_tuned_lr0.05_leaves31_depth5,0.883895,0.835979,0.947439,0.842697,0.781250,0.900267,0.05,31,5,110
3,LGBM_tuned_lr0.03_leaves15_depth5,0.883895,0.837696,0.945830,0.859551,0.803150,0.900000,0.03,15,5,183
23,LGBM_tuned_lr0.1_leaves63_depth5,0.874532,0.820375,0.933931,0.842697,0.781250,0.897594,0.10,63,5,35


## 12. 튜닝 결과 중 최고 성능 모델 확인

각 boosting 모델에서   validation ROC-AUC가 가장 높았던 설정 확인

In [17]:
best_xgb_row = xgb_tuning_df.iloc[0].copy()
best_lgb_row = lgb_tuning_df.iloc[0].copy()

best_tuned_df = pd.DataFrame([best_xgb_row, best_lgb_row]).sort_values("Valid_ROC_AUC", ascending=False)
best_tuned_df

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,learning_rate,max_depth,Best_iteration,num_leaves
6,XGBoost_tuned_lr0.1_depth3,0.900749,0.863049,0.952643,0.859551,0.81203,0.908155,0.10,3,117,NaN
12,LGBM_tuned_lr0.05_leaves15_depth5,0.885768,0.839050,0.948506,0.853933,0.80000,0.903743,0.05,5,117,15.0


## 13. Validation 기반 최종 후보 비교

validation 성능만으로 최종 후보 비교
비교 대상:
- Random Forest baseline
- 튜닝된 XGBoost
- 튜닝된 LightGBM

In [18]:
candidate_rows = pd.DataFrame([
    rf_result,
    best_xgb_row.to_dict(),
    best_lgb_row.to_dict()
])

candidate_rows = candidate_rows[[
    "Model",
    "Train_Accuracy", "Train_F1", "Train_ROC_AUC",
    "Valid_Accuracy", "Valid_F1", "Valid_ROC_AUC"
]].sort_values("Valid_ROC_AUC", ascending=False)

candidate_rows

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC
1,XGBoost_tuned_lr0.1_depth3,0.900749,0.863049,0.952643,0.859551,0.81203,0.908155
2,LGBM_tuned_lr0.05_leaves15_depth5,0.885768,0.839050,0.948506,0.853933,0.80000,0.903743
0,RandomForest_baseline,0.882022,0.830189,0.937260,0.820225,0.75000,0.890575


## 14. Early Stopping 효과 비교
LightGBM 기준으로
1. Early Stopping 미적용
2. Early Stopping 적용

In [19]:
# 같은 설정, early stopping 없음
lgb_no_es = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=2000,
    learning_rate=best_lgb_params["learning_rate"],
    num_leaves=best_lgb_params["num_leaves"],
    max_depth=best_lgb_params["max_depth"],
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

lgb_no_es.fit(X_train_prep, y_train)

# 같은 설정, early stopping 있음
lgb_with_es = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=2000,
    learning_rate=best_lgb_params["learning_rate"],
    num_leaves=best_lgb_params["num_leaves"],
    max_depth=best_lgb_params["max_depth"],
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

lgb_with_es.fit(
    X_train_prep, y_train,
    eval_set=[(X_valid_prep, y_valid)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

,boosting_type,'gbdt'
,num_leaves,15
,max_depth,5
,learning_rate,0.05
,n_estimators,2000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
def evaluate_all_splits(model, model_name):
    train_prob = model.predict_proba(X_train_prep)[:, 1]
    valid_prob = model.predict_proba(X_valid_prep)[:, 1]
    test_prob = model.predict_proba(X_test_prep)[:, 1]

    train_metrics = compute_metrics(y_train, train_prob)
    valid_metrics = compute_metrics(y_valid, valid_prob)
    test_metrics = compute_metrics(y_test, test_prob)

    row = {
        "Model": model_name,
        "Train_Accuracy": train_metrics["Accuracy"],
        "Train_F1": train_metrics["F1"],
        "Train_ROC_AUC": train_metrics["ROC_AUC"],
        "Valid_Accuracy": valid_metrics["Accuracy"],
        "Valid_F1": valid_metrics["F1"],
        "Valid_ROC_AUC": valid_metrics["ROC_AUC"],
        "Test_Accuracy": test_metrics["Accuracy"],
        "Test_F1": test_metrics["F1"],
        "Test_ROC_AUC": test_metrics["ROC_AUC"],
    }
    return row

early_stopping_compare = pd.DataFrame([
    evaluate_all_splits(lgb_no_es, "LightGBM_no_early_stopping"),
    evaluate_all_splits(lgb_with_es, "LightGBM_with_early_stopping"),
]).sort_values("Valid_ROC_AUC", ascending=False)

early_stopping_compare["Best_iteration"] = [
    None,
    lgb_with_es.best_iteration_
]

early_stopping_compare

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC,Test_Accuracy,Test_F1,Test_ROC_AUC,Best_iteration
1,LightGBM_with_early_stopping,0.885768,0.839050,0.948506,0.853933,0.80000,0.903743,0.826816,0.755906,0.833399,NaN
0,LightGBM_no_early_stopping,0.979401,0.972973,0.999014,0.825843,0.77037,0.862968,0.765363,0.700000,0.809816,117.0


## 15. 최종 모델 선택

validation ROC-AUC를 기준으로 최종 제출 후보 모델 선택

In [21]:
models_for_selection = {
    "RandomForest": rf_model,
    "XGBoost_tuned": best_xgb_model,
    "LightGBM_tuned": best_lgb_model,
}

valid_auc_scores = {
    "RandomForest": rf_result["Valid_ROC_AUC"],
    "XGBoost_tuned": float(best_xgb_row["Valid_ROC_AUC"]),
    "LightGBM_tuned": float(best_lgb_row["Valid_ROC_AUC"]),
}

final_model_name = max(valid_auc_scores, key=valid_auc_scores.get)
final_model = models_for_selection[final_model_name]

print("Validation ROC-AUC 기준 최종 선택 모델:", final_model_name)
print("Validation ROC-AUC:", valid_auc_scores[final_model_name])

Validation ROC-AUC 기준 최종 선택 모델: XGBoost_tuned
Validation ROC-AUC: 0.9081550802139038


## 16. 최종 Test 평가

validation에서 선택한 최종 모델을 test 데이터에서 평가.

In [23]:
final_test_result = evaluate_test(final_model, X_test_prep, y_test, final_model_name)
pd.DataFrame([final_test_result])

,Model,Test_Accuracy,Test_F1,Test_ROC_AUC
0,XGBoost_tuned,0.798883,0.723077,0.81087


## 17. 최종 결과 표 정리

In [24]:
summary_valid_table = candidate_rows.copy()
summary_valid_table

,Model,Train_Accuracy,Train_F1,Train_ROC_AUC,Valid_Accuracy,Valid_F1,Valid_ROC_AUC
1,XGBoost_tuned_lr0.1_depth3,0.900749,0.863049,0.952643,0.859551,0.81203,0.908155
2,LGBM_tuned_lr0.05_leaves15_depth5,0.885768,0.839050,0.948506,0.853933,0.80000,0.903743
0,RandomForest_baseline,0.882022,0.830189,0.937260,0.820225,0.75000,0.890575


In [25]:
summary_test_table = pd.DataFrame([final_test_result])
summary_test_table

,Model,Test_Accuracy,Test_F1,Test_ROC_AUC
0,XGBoost_tuned,0.798883,0.723077,0.81087


## 18. 분석 정리

이번 실험에서 validation ROC-AUC 기준 최종 선택 모델은 튜닝된 XGBoost였다.
최적 설정은 **learning_rate=0.1, max_depth=3**였고, validation ROC-AUC는 0.9082였다.

다만 baseline만 놓고 보면 LightGBM baseline이 가장 높았고 early stopping 비교에서는 LightGBM에서 early stopping 효과가 매우 크게 나타났다.

이번 과제의 실험 결과를 바탕으로
	•	Random Forest는 안정적인 baseline
	•	Boosting은 더 높은 성능을 낼 수 있지만, validation 기반 제어가 필수
라는 6주차 강의 내용을 다시 한번 확인할 수 있었다.